In [ ]:
# =========================
# RQ5: Model Ranking Comparison (CLEAN VERSION)
# =========================

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb

# -------------------------
# 1. Load Data
# -------------------------
df = pd.read_csv(
    "/kaggle/input/datasets/sharmajicoder/gaming-and-mental-health/gaming_mental_health_10M_40features.csv"
)

df = df.dropna()
df = df.sample(n=20000, random_state=42)

TARGET = df.columns[-1]

# -------------------------
# 2. Encode Target
# -------------------------
le = LabelEncoder()
y_raw = le.fit_transform(df[TARGET])

median_value = np.median(y_raw)
y = (y_raw > median_value).astype(int)

print(pd.Series(y).value_counts())

# -------------------------
# 3. Features
# -------------------------
X = df.drop(TARGET, axis=1)
X = pd.get_dummies(X, drop_first=True)

# -------------------------
# 4. Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# 5. Scaling (for SVM only)
# -------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------
# 6. Models
# -------------------------
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

# -------------------------
# 7. Train & Evaluate
# -------------------------
results = []

for name, model in models.items():
    print(f"Training {name}...")

    if name == "SVM":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")
    # auc = roc_auc_score(y_test, y_prob, multi_class="ovr")
    auc = roc_auc_score(y_test, y_prob[:, 1])

    results.append([name, acc, rec, f1, auc])

# -------------------------
# 8. Ranking Table (NO METRICS TABLE)
# -------------------------
ranking_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Recall", "F1", "AUC"
])

for metric in ["Accuracy", "Recall", "F1", "AUC"]:
    ranking_df[metric + "_Rank"] = ranking_df[metric].rank(ascending=False)

ranking_df.to_csv("RQ5_ranking_table.csv", index=False)

print("\n=== Ranking Table ===")
print(ranking_df)

# -------------------------
# 9. Ranking Line Plot (MAIN FIGURE)
# -------------------------
rank_plot = ranking_df.set_index("Model")[[
    "Accuracy_Rank",
    "Recall_Rank",
    "F1_Rank",
    "AUC_Rank"
]]

metrics = ["Accuracy", "Recall", "F1-score", "AUC"]

plt.figure(figsize=(8, 5))

markers = ['o', 's', '^']

for i, model in enumerate(rank_plot.index):
    ranks = rank_plot.loc[model].values
    plt.plot(metrics, ranks, marker=markers[i], label=model)

    # Add rank numbers
    for j, value in enumerate(ranks):
        plt.text(j, value, str(int(value)), ha='center', va='bottom')

# Rank 1 at top
# plt.gca().invert_yaxis()

plt.title("RQ5: Sensitivity to Evaluation Metrics")
plt.xlabel("Evaluation Metric")
plt.ylabel("Rank (1 = Best)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("RQ5_ranking_line_figure.pdf")
plt.show()